In [ ]:
import pandas as pd
import seaborn as sns
pd.options.display.precision = 2
import matplotlib.pyplot as plt
import json
import sys

In [ ]:
sys.path.append('../')

In [ ]:
League_path = '/Users/cmartin/Fantasy_Baseball/Ottoneu_Baseball_Projects/Imaginary_Hammers/'

In [ ]:
Calc_date = 'Mar21_2026'

In [ ]:
my_TeamID = 240.

In [ ]:
All_Teams_csv_path = League_path+'/Rosters'+f'/Kept_Players_and_Replacement_Level_{Calc_date}.csv'

In [ ]:
All_Teams_df = pd.read_csv(All_Teams_csv_path)
All_Teams_df['FG ID'] = All_Teams_df['FG ID'].astype(str)
All_Teams_df['FG ID'] = All_Teams_df['FG ID'].str.replace('.0','')
All_Teams_df['Ottoneu ID'] = All_Teams_df['Ottoneu ID'].astype(str)
All_Teams_df['Ottoneu ID'] = All_Teams_df['Ottoneu ID'].str.replace('.0','')

In [ ]:
Target_Stats_path = League_path+'/Target_Stats_dict.json'

In [ ]:
Additional_targets = {
    'G_mySGP_Team':162.*12.,
    'G_FGAV_Team': 162.*12.,
    'IP_mySGP_Team':1500.,
    'IP_FGAV_Team':1500.,
    'TOTAL_SGP_Val_mySGP':500., #terget value for full team
    'R_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'HR_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'OBP_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'SLG_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'SO_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'HR9_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'ERA_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'WHIP_SGP_Val_mySGP_Team':200., # Not exactly correct, but OK for now. See Cells 115 & 125 in Player SGP
    'Salary': 380. #assume 10 for in-season pickups
}

Batting_value_columns = {
    'G':['TOTAL_SGP_Val_mySGP','G_mySGP_Team','PA_mySGP_Team','PA_FGAV_Team','Salary'],
    'R':['R_mySGP_Team','R_SGP_Val_mySGP_Team'], #'R_SGP_raw_mySGP','R_SGP_norm_mySGP','mR_FGAV'
    'HR':['HR_mySGP_Team','HR_SGP_Val_mySGP_Team'], # 'HR_SGP_raw_mySGP','HR_SGP_norm_mySGP','mHR_FGAV'
    'OBP':['OBP_Team','OBP_SGP_Val_mySGP_Team'], # 'OBP_SGP_raw_mySGP','OBP_SGP_norm_mySGP','mOBP_FGAV'
    'SLG':['SLG_Team','SLG_SGP_Val_mySGP_Team'] #'SLG_SGP_raw_mySGP','SLG_SGP_norm_mySGP',,'mSLG_FGAV'
}

Pitching_value_columns = {
    'IP':['TOTAL_SGP_Val_mySGP','IP_mySGP_Team','IP_FGAV_Team','Salary'],
    'SO':['SO_mySGP_Team','SO_SGP_Val_mySGP_Team'], # 'SO_SGP_raw_mySGP','SO_SGP_norm_mySGP',,'mSO_FGAV'
    'HR9':['HR9_Team','HR9_SGP_Val_mySGP_Team'], #'HR9_SGP_raw_mySGP','HR9_SGP_norm_mySGP', ,'mHR_FGAV'
    'ERA':['ERA_Team','ERA_SGP_Val_mySGP_Team'], # 'ERA_SGP_raw_mySGP','ERA_SGP_norm_mySGP',,'mERA_FGAV'
    'WHIP':['WHIP_Team','WHIP_SGP_Val_mySGP_Team'] # 'WHIP_SGP_raw_mySGP','WHIP_SGP_norm_mySGP',,'mWHIP_FGAV'
}

Team_Rate_Cols = [
    'HR9_Team',
    'ERA_Team',
    'WHIP_Team',
    'OBP_Team',
    'SLG_Team'
]

In [ ]:
with open(Target_Stats_path, 'r') as f:
    data = json.load(f)
    data.update(Additional_targets)
Target_Stats_df = pd.json_normalize(data)

In [ ]:
Target_Stats_df.columns

In [ ]:
Target_Stats_Hitter_df = Target_Stats_df[['Target Pts','R','HR','OBP','SLG','G_mySGP_Team','G_FGAV_Team','TOTAL_SGP_Val_mySGP','R_SGP_Val_mySGP_Team', 'HR_SGP_Val_mySGP_Team', 'OBP_SGP_Val_mySGP_Team', 'SLG_SGP_Val_mySGP_Team','Salary']]
Target_Stats_Hitter_df['Hitter Pitcher'] = 'Hitter'
Target_Stats_Hitter_df.rename(columns={'R':'R_mySGP_Team','HR':'HR_mySGP_Team','OBP':'OBP_Team','SLG':'SLG_Team'},inplace=True)
Target_Stats_Pitcher_df = Target_Stats_df[['Target Pts','K','HR9','ERA','WHIP','IP_mySGP_Team','IP_FGAV_Team','TOTAL_SGP_Val_mySGP','SO_SGP_Val_mySGP_Team', 'HR9_SGP_Val_mySGP_Team', 'ERA_SGP_Val_mySGP_Team', 'WHIP_SGP_Val_mySGP_Team','Salary']]
Target_Stats_Pitcher_df['Hitter Pitcher'] = 'Pitcher'
Target_Stats_Pitcher_df.rename(columns={'K':'SO_mySGP_Team','HR9':'HR9_Team','ERA':'ERA_Team','WHIP':'WHIP_Team'},inplace=True)
#Target_Stats_Pitcher_df[] = 
Target_Stats_df = pd.concat([
    Target_Stats_Hitter_df,
    Target_Stats_Pitcher_df
])

In [ ]:
Target_Stats_df.head()

In [ ]:
Player_id_cols = [
    'FG ID','Name','Team','Ottoneu ID','Ottoneu Positions'
]

In [ ]:
Fantasy_Team_ID_cols = [
    'TeamID', 'Team Name',
]

In [ ]:
Games_IP_Targets = {
    'C':162,
    '1B':162,
    '2B':162,
    '3B':162,
    'SS':162,
    'MI':162, 
    'OF':810, #total
    'Util':162, 
    'IP':1500 #total
}
Start_Spots = {
    'C':1, #technically 2 but 162 G limit
    '1B':1,
    '2B':1,
    'SS':1,
    'MI':1,
    '3B':1,
    'OF':5,
    'Util':1, #starting Hitters
    'SP':5,
    'RP':5
}

Total_Roster_Spots = {
    'C':2,
    '1B':2,
    '2B':3,
    'SS':3,
    '3B':2,
    'OF':9,
    'SP':9,
    'RP':6
}

In [ ]:
#Scoring Categories
Count_Scoring_Categories_Batting = [
    'R',
    'HR'
]
Rate_Scoring_Categories_Batting = [
    'OBP',
    'SLG'
]
Count_Scoring_Categories_Pitching = [
    'SO'
]
Rate_Scoring_Categories_Pitching = [
    "HR9",
    "ERA",
    "WHIP"
]
Num_teams = 12.
Team_budget = 400.
Hitter_sal_split = 0.53
League_budget = Team_budget*Num_teams
Hitter_budget = League_budget*Hitter_sal_split
Pitcher_budget = League_budget*(1.-Hitter_sal_split)
Scoring_Categories_Batting = Count_Scoring_Categories_Batting + Rate_Scoring_Categories_Batting
Scoring_Categories_Pitching = Count_Scoring_Categories_Pitching + Rate_Scoring_Categories_Pitching
Scoring_Categories = Scoring_Categories_Batting + Scoring_Categories_Pitching

In [ ]:
Hitting_Pos = [
    'C',
    '1B',
    '2B',
    'SS',
    '3B',
    'OF',
    'MI',
    'Util'
]
Pitching_Pos = [
    'SP',
    'RP'
]

All_Pos = Hitting_Pos + Pitching_Pos

In [ ]:
def quick_plotting_fn(quick_plot, title):
    fig = plt.figure(figsize=(12,6))
    ax1 = fig.add_subplot(111)
    ax1.errorbar(
        y=quick_plot['Name'],
        x=quick_plot['Ottoneu_Avg'],
        xerr=[
            quick_plot['Ottoneu_Avg']-quick_plot['Ottoneu_Min'],
            quick_plot['Ottoneu_Max']-quick_plot['Ottoneu_Avg']
        ],
        fmt='o',
        color='blue',
        label='4x4 Avg, Min, Max'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Ottoneu_Med'],
        marker='^',
        color='black',
        label='4x4 Median'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Salary'],
        marker='$\\$$',
        s=150,
        color='red',
        label='Current Salary'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Dollars_FGAV'],
        marker='$FG$',
        s=150,
        color='green',
        label='FG Auction Calc.'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['Dollars_Vibbot'],
        marker='+',
        s=100,
        color='tab:brown',
        label='Secret Sauce V'
    )
    ax1.scatter(
        y=quick_plot['Name'],
        x=quick_plot['TOTAL_SGP_Val_mySGP'],
        marker='*',
        s=100,
        color='tab:pink',
        label='Secret Sauce C'
    )
    ax1.set_ylabel(title)
    ax1.invert_yaxis()
    ax1.set_xlabel('Dollars')
    ax1.legend(loc='lower right')
    plt.tick_params(axis='y', which='major', labelsize=7)
    plt.show()

In [ ]:
for team_id in All_Teams_df['TeamID'].unique():
    This_team_df = All_Teams_df[All_Teams_df['TeamID'] == team_id]
    team_name = All_Teams_df[All_Teams_df['TeamID'] == team_id]['Team Name'].unique()[0]
    print(team_name)
    quick_plotting_fn(This_team_df,team_name)

In [ ]:
for team_id in All_Teams_df['TeamID'].unique():
    This_team_df = All_Teams_df[All_Teams_df['TeamID'] == team_id]
    team_name = All_Teams_df[All_Teams_df['TeamID'] == team_id]['Team Name'].unique()[0]
    #print(team_name)
    display(This_team_df.loc[This_team_df["Name"].str.startswith('Replacement', na=False)][['Team Name','Ottoneu Positions','G_mySGP_Team','IP_mySGP_Team']])
    #quick_plotting_fn(This_team_df,team_name)

In [ ]:
for pos in All_Teams_df.loc[All_Teams_df["Name"].str.startswith('Replacement', na=False)]['Ottoneu Positions'].unique():
    This_pos_df = All_Teams_df[All_Teams_df['Ottoneu Positions'] == pos]
    #print(team_name)
    display(This_pos_df.loc[This_pos_df["Name"].str.startswith('Replacement', na=False)][['Team Name','Ottoneu Positions','G_mySGP_Team','IP_mySGP_Team']].reset_index(drop=True).sort_values(by=['G_mySGP_Team','IP_mySGP_Team'],ascending=False))
    #quick_plotting_fn(This_team_df,team_name)

In [ ]:
%%capture cap0 --no-stderr
for pos in All_Teams_df.loc[All_Teams_df["Name"].str.startswith('Replacement', na=False)]['Ottoneu Positions'].unique():
    This_pos_df = All_Teams_df[All_Teams_df['Ottoneu Positions'] == pos]
    print('+++++++++++++++++++++++++++++++++++++++++++++++++++')
    print('+++++++++++++++++++++++++++++++++++++++++++++++++++\n')
    print(pos,'\n')
    print('+++++++++++++++++++++++++++++++++++++++++++++++++++')
    Repl_df = This_pos_df.loc[This_pos_df["Name"].str.startswith('Replacement', na=False)][['Team Name','Ottoneu Positions','G_mySGP_Team','IP_mySGP_Team']].reset_index(drop=True).sort_values(by=['G_mySGP_Team','IP_mySGP_Team'],ascending=False)
    print('Est. Demand:', Repl_df.shape[0])
    print('My Need:', Repl_df[Repl_df['Team Name'] == 'Largely Indistinguishables'].shape[0])
    #quick_plotting_fn(This_team_df,team_name)

In [ ]:
with open('Team_Est_Demand.txt', 'w') as f:
    f.write(cap0.stdout)

In [ ]:
%%capture cap --no-stderr
for pos in ['SP','Util']:
    This_Pos_Stat_Cats = Pitching_value_columns if pos in Pitching_Pos else Batting_value_columns
    print('+++++++++++++++++++++++++++++++++++++++++++++++++++')
    print('+++++++++++++++++++++++++++++++++++++++++++++++++++\n\n')
    print(pos,'\n\n')
    print('+++++++++++++++++++++++++++++++++++++++++++++++++++')
    print('+++++++++++++++++++++++++++++++++++++++++++++++++++')
    for stat_cat, stat_cols in This_Pos_Stat_Cats.items():
        # if Stat in bugged_cols:
        #     continue
        # print(pos)
        # print(stat_cat)
        # print(All_teams_replacement_level[stat_cat])
        # fig1 = plt.figure(figsize=(10,5))
        # ax1 = fig1.add_subplot(111)
        # sns.histplot(All_teams_replacement_level,x='Team Name',weights=stat_cat,hue='Ottoneu Positions', multiple='stack',ax=ax1)
        # ax1.axhline(y=Target_Stats_df[stat_cat])
        # plt.tight_layout()
        # plt.show()

        print('+++++++++++++++++++++++++++++++++++++++++++++++++++\n')
        print(stat_cat,'\n')
        print('+++++++++++++++++++++++++++++++++++++++++++++++++++')
        
        

        for stat in stat_cols:
            print('---------------------------------------------------')
            print(stat)
            print('---------------------------------------------------')
            # fig2 = plt.figure(figsize=(10,5))
            # ax2 = fig2.add_subplot(111)
            my_team_currently = 0.
            if stat in Team_Rate_Cols:
                # sns.histplot(All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).groupby('Team Name')[stat].mean().sort_values().reset_index(),y='Team Name',weights=stat,color='darkorange',ax=ax2)
                # sns.scatterplot(All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat.replace('_','_pos_')),y='Team Name',x=stat.replace('_','_pos_'),hue='Repl_Pos_mySGP',ax=ax2)
                team_mean_df = All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).groupby('Team Name')[stat].mean().sort_values().reset_index()
                my_team_currently = team_mean_df[team_mean_df['Team Name'] == 'Largely Indistinguishables'][stat].unique()[0]
                team_mean_df['rank'] = team_mean_df[stat].rank(ascending=False)
                #display(team_mean_df)
                my_team_rank = team_mean_df[team_mean_df['Team Name'] == 'Largely Indistinguishables']['rank'].unique()[0]
            else:
                #sns.histplot(All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).sort_values(stat),y='Team Name',weights=stat,hue='Repl_Pos_mySGP', multiple='stack',ax=ax2)
                team_sum_df = All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).groupby('Team Name')[stat].sum().sort_values().reset_index()
                my_team_currently = team_sum_df[team_sum_df['Team Name'] == 'Largely Indistinguishables'][stat].unique()[0]
                team_sum_df['rank'] = team_sum_df[stat].rank(ascending=False)
                #display(team_sum_df)
                my_team_rank = team_sum_df[team_sum_df['Team Name'] == 'Largely Indistinguishables']['rank'].unique()[0]
            # ax2.tick_params(axis='x', labelrotation=45)
            # ax2.set_xlabel(stat)
            # ax2.set_ylabel('Team Name')
            if (stat in Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')].dropna(axis=1).columns):
                #ax2.axvline(x=Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')][stat].unique()[0],label='Target')
                print(f'Target {stat}:', Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')][stat].unique()[0])
                print('Rank with Replacement Players:',my_team_rank)
                print('Largely Indistinguishables with Replacement Players:',my_team_currently)
                print(f'Delta:', my_team_currently - Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')][stat].unique()[0])
            else:
                print(Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')].dropna(axis=1).columns)
            # plt.tight_layout()
            # plt.show()
            #sns.histplot(All_teams_replacement_level.dropna(subset=stat),y=stat,hue=)

In [ ]:
with open('Team_Ranks_with_Replacement.txt', 'w') as f:
    f.write(cap.stdout)

In [ ]:
for pos in ['SP','Util']:
    This_Pos_Stat_Cats = Pitching_value_columns if pos in Pitching_Pos else Batting_value_columns
    for stat_cat, stat_cols in This_Pos_Stat_Cats.items():
        # if Stat in bugged_cols:
        #     continue
        # print(pos)
        # print(stat_cat)
        # print(All_teams_replacement_level[stat_cat])
        # fig1 = plt.figure(figsize=(10,5))
        # ax1 = fig1.add_subplot(111)
        # sns.histplot(All_teams_replacement_level,x='Team Name',weights=stat_cat,hue='Ottoneu Positions', multiple='stack',ax=ax1)
        # ax1.axhline(y=Target_Stats_df[stat_cat])
        # plt.tight_layout()
        # plt.show()
        
        

        for stat in stat_cols:
            # print(stat)
            fig2 = plt.figure(figsize=(10,5))
            ax2 = fig2.add_subplot(111)
            my_team_currently = 0.
            if stat in Team_Rate_Cols:
                sns.histplot(All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).groupby('Team Name')[stat].mean().sort_values().reset_index(),y='Team Name',weights=stat,color='darkorange',ax=ax2)
                sns.scatterplot(All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat.replace('_','_pos_')),y='Team Name',x=stat.replace('_','_pos_'),hue='Repl_Pos_mySGP',ax=ax2)
                team_mean_df = All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).groupby('Team Name')[stat].mean().sort_values().reset_index()
                team_mean_df['rank'] = team_mean_df[stat].rank(ascending=False)
                display(team_mean_df)
                my_team_currently = team_mean_df[team_mean_df['Team Name'] == 'Largely Indistinguishables'][stat].unique()[0]
            else:
                sns.histplot(All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).sort_values(stat),y='Team Name',weights=stat,hue='Repl_Pos_mySGP', multiple='stack',ax=ax2)
                team_sum_df = All_Teams_df[All_Teams_df['Repl_Pos_mySGP'].isin(Pitching_Pos if pos in Pitching_Pos else Hitting_Pos) if stat not in ['TOTAL_SGP_Val_mySGP','Salary'] else [True]*len(All_Teams_df)].dropna(subset=stat).groupby('Team Name')[stat].sum().sort_values().reset_index()
                team_sum_df['rank'] = team_sum_df[stat].rank(ascending=False)
                display(team_sum_df)
                my_team_currently = team_sum_df[team_sum_df['Team Name'] == 'Largely Indistinguishables'][stat].unique()[0]
            ax2.tick_params(axis='x', labelrotation=45)
            ax2.set_xlabel(stat)
            ax2.set_ylabel('Team Name')
            if (stat in Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')].dropna(axis=1).columns):
                ax2.axvline(x=Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')][stat].unique()[0],label='Target')
                print(f'Target {stat}:', Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')][stat].unique()[0])
                print('Largely Indistinguishables with Replacement Players:',my_team_currently)
                print(f'Delta:', my_team_currently - Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')][stat].unique()[0])
            else:
                print(Target_Stats_df[Target_Stats_df['Hitter Pitcher'] == ('Pitcher' if pos in Pitching_Pos else 'Hitter')].dropna(axis=1).columns)
            plt.tight_layout()
            plt.show()
            #sns.histplot(All_teams_replacement_level.dropna(subset=stat),y=stat,hue=)